# S3.13 — when/otherwise
**Date completed:** October 2026  
**Status:** Completed  
**Key concept:** PySpark equivalent of SQL CASE WHEN — order matters

In [0]:
# ============================================================
# S3.13 — Cell 2
# when() / otherwise(): Conditional Column Logic
#
# Goal:
# 1. Use basic when()/otherwise() for a single condition.
# 2. Chain multiple when() conditions for categorisation.
# 3. Understand that condition order matters.
# 4. Handle NULL values explicitly.
# 5. Use multiple-column conditions with AND / OR.
# 6. Understand nested when()/otherwise().
# 7. Understand behaviour when otherwise() is omitted.
# 8. Apply conditional logic to samples.tpch.orders.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Important:
# when()/otherwise() is Spark's DataFrame expression pattern
# corresponding conceptually to SQL CASE WHEN.
# ============================================================

import pyspark.sql.functions as F

print("=== S3.13: when() / otherwise() ===")
print()

# ------------------------------------------------------------
# Step 1 — Load source table
# ------------------------------------------------------------

df_orders = spark.table(
    "samples.tpch.orders"
)

print("Orders DataFrame loaded.")
print()

# ------------------------------------------------------------
# Step 2 — Basic when() / otherwise()
#
# Business requirement:
#
# Convert compact order-status codes into readable labels.
#
# SQL equivalent conceptually:
#
# CASE
#     WHEN status = 'F' THEN 'Fulfilled'
#     WHEN status = 'O' THEN 'Open'
#     WHEN status = 'P' THEN 'Pending'
#     ELSE 'Unknown'
# END
# ------------------------------------------------------------

print("=== STEP 2: BASIC STATUS MAPPING ===")


df_status = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status_code"),

    F.when(F.col("o_orderstatus") == "F","Fulfilled")

    .when(F.col("o_orderstatus") == "O","Open")

    .when(F.col("o_orderstatus") == "P","Pending")

    .otherwise("Unknown").alias("status_label"))

df_status.show(10, truncate=False)

print("\nStatus distribution:")

(
    df_status
    .groupBy("status_code","status_label")

    .agg(F.count("*").alias("order_count"))

    .orderBy("status_code")

    .show(truncate=False)
)

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Multiple when() Conditions
#
# Business requirement:
#
# Categorise orders by total value.
#
# IMPORTANT:
# Conditions are evaluated from top to bottom.
#
# First matching condition wins.
#
# Therefore ranges must be ordered carefully.
# ------------------------------------------------------------

print("=== STEP 3: RANGE CATEGORISATION ===")


df_segments = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_totalprice").alias("total_price"),

    F.when(F.col("o_totalprice").isNull(),"Unknown")

    .when(F.col("o_totalprice") < 10000,"Low Value")

    .when(F.col("o_totalprice") < 50000,"Medium Value")

    .when(F.col("o_totalprice") < 100000, "High Value")

    .otherwise("Premium").alias("value_segment"))

df_segments.show(10,truncate=False)

print("\nValue-segment distribution:")

(
    df_segments
    .groupBy("value_segment")

    .agg(F.count("*").alias("order_count"),
         F.round(F.avg("total_price"),2).alias("avg_price"))

    .orderBy("avg_price")
    
    .show(truncate=False))

print()


In [0]:
# ------------------------------------------------------------
# Step 4 — Why Condition Order Matters
#
# Our logic means:
#
# price < 10,000
#     → Low
#
# otherwise price < 50,000
#     → Medium
#
# otherwise price < 100,000
#     → High
#
# otherwise
#     → Premium
#
# Example:
#
# price = 8,000
#
# It satisfies BOTH:
#
# price < 10,000
# price < 50,000
#
# But the first TRUE condition wins.
#
# Therefore it becomes "Low Value".
# ------------------------------------------------------------

print("=== STEP 4: FIRST MATCH WINS ===")

print("""
when() conditions are evaluated from top to bottom.

The first condition that evaluates to TRUE supplies the result.

Example:

price = 8,000

price < 10,000   → TRUE
price < 50,000   → also TRUE

But Spark stops at the first matching branch.

Result:
Low Value

Therefore condition ordering is part of the business logic.
""")


In [0]:
# ------------------------------------------------------------
# Step 5 — Multiple Column Conditions
#
# PySpark Boolean operators:
#
# & → AND
# | → OR
# ~ → NOT
#
# Always use parentheses around individual Column conditions.
# ------------------------------------------------------------

print("=== STEP 5: MULTIPLE COLUMN CONDITIONS ===")


df_priority = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status"),

    F.col("o_totalprice").alias("total_price"),

    F.when(F.col("o_orderstatus").isNull() |  F.col("o_totalprice").isNull(),"UNKNOWN")

    .when((F.col("o_orderstatus") == "O") & (F.col("o_totalprice") > 100000), "URGENT - High value open order")

    .when((F.col("o_orderstatus") == "O") & (F.col("o_totalprice") > 50000), "MONITOR - Medium value open order")

    .when(F.col("o_orderstatus") == "F", "CLOSED - Fulfilled")

    .otherwise("NORMAL") .alias("priority_flag"))

print("\nPriority-flag distribution:")

(
    df_priority

    .groupBy("priority_flag")

    .agg(F.count("*").alias("order_count"))

    .orderBy(F.col("order_count").desc())

    .show(truncate=False))

print()



In [0]:
# ------------------------------------------------------------
# Step 6 — OR Condition Example
#
# Requirement:
#
# Identify orders whose status is either:
#
# F
# OR
# O
# ------------------------------------------------------------

print("=== STEP 6: OR CONDITION ===")

df_status_group = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status"),

    F.when((F.col("o_orderstatus") == "F") | (F.col("o_orderstatus") == "O"), "Known Active/Completed Status")

    .otherwise("Other Status").alias("status_group"))

df_status_group.show( 10, truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 7 — Nested when() / otherwise()
#
# Chained when() is normally easier to read for simple
# CASE-WHEN style categorisation.
#
# Nested when() is still possible when one decision contains
# another conditional decision.
#
# Requirement:
#
# If status is Open:
#     If price > 100000 → High Priority Open
#     Else              → Standard Open
#
# Otherwise:
#     → Not Open
# ------------------------------------------------------------

print("=== STEP 7: NESTED when() ===")

df_nested = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status"),

    F.col("o_totalprice").alias("total_price"),

    F.when(F.col("o_orderstatus") == "O",
           F.when(F.col("o_totalprice") > 100000,"High Priority Open").otherwise("Standard Open"))

    .otherwise("Not Open").alias("nested_category"))

df_nested.show(10, truncate=False)

print("""
Nested when() is valid.

However, for many categorisation problems,
chained when() statements are easier to read and maintain.

Use nesting only when the business decision itself is hierarchical.
""")


In [0]:
# ------------------------------------------------------------
# Step 8 — when() Without otherwise()
#
# If no condition matches and otherwise() is omitted,
# the resulting value is NULL.
# ------------------------------------------------------------

print("=== STEP 8: when() WITHOUT otherwise() ===")

df_no_otherwise = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status"),

    F.when(F.col("o_orderstatus") == "F","Fulfilled")

    .alias("fulfilled_only"))

print("\nDistribution without otherwise():")

(
    df_no_otherwise

    .groupBy("fulfilled_only").agg(F.count("*").alias("order_count"))

    .show(truncate=False))

print("""
If:

status == "F"
→ "Fulfilled"

Otherwise:
→ NULL

because no otherwise() branch was supplied.
""")


In [0]:
# ------------------------------------------------------------
# Step 9 — NULL Condition Behaviour
#
# Important SQL three-valued logic:
#
# TRUE
# FALSE
# UNKNOWN / NULL
#
# A when() branch executes only when its condition is TRUE.
#
# If the condition evaluates to NULL/UNKNOWN,
# Spark does not treat it as TRUE.
#
# Evaluation continues to later branches / otherwise().
# ------------------------------------------------------------

print("=== STEP 9: NULL CONDITION BEHAVIOUR ===")


null_test_data = [(1, 150000.0), (2, 5000.0),(3, None)]

df_null_test = spark.createDataFrame(null_test_data,["id", "price"])

df_null_when = df_null_test.select(

    "id",
    "price",
    F.when(F.col("price") > 100000,"High").otherwise("Not High").alias("price_category"))

df_null_when.show(truncate=False)

print("""
For price = NULL:

price > 100000
→ NULL / UNKNOWN

The condition is not TRUE.

Therefore Spark moves to otherwise().

This is why important NULL cases should often be handled explicitly.
""")

In [0]:
# ------------------------------------------------------------
# Step 10 — Explicit NULL Handling Pattern
#
# Better pattern when NULL has separate business meaning:
#
# NULL
#     → Unknown
#
# < 10,000
#     → Low
#
# otherwise
#     → High
# ------------------------------------------------------------

print("=== STEP 10: EXPLICIT NULL HANDLING ===")

df_null_handled = df_null_test.select(

    "id",

    "price",

    F.when(F.col("price").isNull(),"Unknown")

    .when(F.col("price") < 10000,"Low")

    .otherwise("High")

    .alias("price_segment"))

df_null_handled.show(
    truncate=False
)

print()


In [0]:
# ------------------------------------------------------------
# Step 11 — Output Data Types
#
# Every branch contributes values to one output column.
#
# Therefore branch outputs should be logically compatible.
#
# Good:
#
# "Low"
# "Medium"
# "High"
#
# All STRING.
#
# Also good:
#
# 1
# 2
# 3
#
# All numeric.
#
# Avoid mixing unrelated output types unless type coercion is
# intentionally understood.
# ------------------------------------------------------------

print("=== STEP 11: OUTPUT TYPE CONSISTENCY ===")


df_type_example = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.when(F.col("o_totalprice") >= 100000,1).otherwise(0).alias("high_value_flag"))

df_type_example.printSchema()

df_type_example.show(
    5,
    truncate=False
)

print()


In [0]:
# ------------------------------------------------------------
# Step 12 — Real TPC-H Business Segmentation
#
# Combine:
#
# status
# total price
# NULL handling
#
# into a Silver-style derived business column.
# ------------------------------------------------------------

print("=== STEP 12: SILVER-STYLE ORDER SEGMENTATION ===")

df_order_segment = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderstatus").alias("status"),

    F.col("o_totalprice").alias("total_price"),

    F.when(F.col("o_totalprice").isNull(),"Unknown")

    .when(F.col("o_totalprice") < 10000,"Low")

    .when(F.col("o_totalprice") < 50000,"Medium")

    .when(F.col("o_totalprice") < 100000,"High").otherwise("Premium").alias("order_value_segment"),

    F.when(F.col("o_orderstatus") == "F",True)
    
    .otherwise(False).alias("is_fulfilled"))

df_order_segment.show(
    10,
    truncate=False
)

print()


In [0]:
# ------------------------------------------------------------
# Step 13 — Distribution Validation
#
# Validate that every source row received exactly one
# segmentation result.
# ------------------------------------------------------------

print("=== STEP 13: VALIDATION ===")

source_count = df_orders.count()

segmented_count = df_order_segment.count()

assert source_count == segmented_count, ("Row count changed during conditional transformation")

null_segment_count = ( df_order_segment.filter(F.col("order_value_segment").isNull()).count())

assert null_segment_count == 0, ("Unexpected NULL order-value segment")

print(f"Source rows    : {source_count:,}")

print(f"Segmented rows : {segmented_count:,}")

print("Conditional transformation validation passed.")

print()


In [0]:
# ------------------------------------------------------------
# Step 14 — when() vs filter()
#
# Important distinction:
#
# when()
#     → Creates / derives a column value.
#
# filter()
#     → Removes rows that do not satisfy a condition.
#
# These solve different problems.
# ------------------------------------------------------------

print("=== STEP 14: when() vs filter() ===")

print("""
when():

Used to DERIVE a value.

Example:

price
  ↓
Low / Medium / High


filter():

Used to KEEP or REMOVE rows.

Example:

Keep only:

price > 100000

Easy memory:

when()
→ Which VALUE should this row receive?

filter()
→ Should this ROW remain?
""")


In [0]:
# ------------------------------------------------------------
# Step 15 — Transformations vs Actions
# ------------------------------------------------------------

print(
    "=== STEP 15: TRANSFORMATIONS VS ACTIONS ==="
)


print("""
Column-expression functions used:

when()
otherwise()

Boolean expressions used:

==
<
>
>=
&
|
isNull()

These create Column expressions.


Transformations used:

select()
filter()
groupBy()
agg()
orderBy()


Actions used:

show()
count()

Spark still follows lazy evaluation.
""")


In [0]:
# ------------------------------------------------------------
# Step 16 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
when():

Defines a conditional Column expression.

Example:

when(
    condition,
    value_if_true
)


otherwise():

Defines the fallback value when none of the preceding
when() conditions evaluate to TRUE.


Multiple when():

Conditions are evaluated from TOP to BOTTOM.

FIRST TRUE condition wins.


Nested when():

A when() expression can contain another when() expression.

Useful for hierarchical decisions, although chained when()
is often easier to read.


Multiple conditions:

& → AND
| → OR
~ → NOT

Use parentheses around individual Column conditions.


NULL behaviour:

A condition evaluating to NULL / UNKNOWN is not TRUE.

Spark continues to the next branch or otherwise().


Without otherwise():

Non-matching rows receive NULL.


Output types:

Branches contribute to one output column.

Keep branch result types logically compatible.


SQL equivalent:

when() / otherwise()

conceptually corresponds to:

CASE
    WHEN ...
    THEN ...
    ELSE ...
END


Critical distinction:

when()
→ derives column VALUES.

filter()
→ removes ROWS.


Silver-layer pattern:

Source columns
      ↓
Validate NULLs
      ↓
Apply ordered business conditions
      ↓
Derive classification / flag
      ↓
Validate distribution
      ↓
Silver-ready business attributes
""")

print("=== S3.13 CELL 2 COMPLETED ===")

# S3.13 — Key Takeaways: when() / otherwise()

## 1. Basic Structure

```python
F.when(
    condition1,
    value1
)
.when(
    condition2,
    value2
)
.when(
    condition3,
    value3
)
.otherwise(
    default_value
)
```

Conceptually:

```text
if condition1:
    value1
elif condition2:
    value2
elif condition3:
    value3
else:
    default_value
```

---

# 2. Critical Rules

1. Conditions are evaluated **top to bottom**.

2. The **first condition that evaluates to TRUE wins**.

3. If a condition evaluates to `NULL / UNKNOWN`, it is not considered TRUE.
   Spark continues to the next `when()` branch or `otherwise()`.

4. If `otherwise()` is omitted, rows that do not match any condition
   receive `NULL`.

5. Branch result values should have logically compatible data types.

Example:

```text
"Low"
"Medium"
"High"
```

are all strings.

Or:

```text
0
1
```

are numeric values.

---

# 3. First Match Wins

Example:

```python
F.when(
    F.col("price") < 10000,
    "Low"
)
.when(
    F.col("price") < 50000,
    "Medium"
)
.otherwise(
    "High"
)
```

For:

```text
price = 8000
```

both of these are mathematically true:

```text
price < 10000
price < 50000
```

But Spark uses the first matching condition.

Result:

```text
Low
```

Therefore:

**Condition order is part of the business logic.**

---

# 4. Multiple Conditions

PySpark Boolean operators:

| Operator | Meaning |
|---|---|
| `&` | AND |
| `\|` | OR |
| `~` | NOT |

Example:

```python
F.when(
    (F.col("status") == "O")
    &
    (F.col("total_price") > 100000),
    "URGENT"
)
```

### Important

Use parentheses around individual Column conditions.

---

# 5. NULL Handling

A comparison involving NULL can evaluate to:

```text
NULL / UNKNOWN
```

Example:

```python
F.col("price") > 100000
```

If:

```text
price = NULL
```

the condition does not evaluate to TRUE.

Spark continues to the next branch.

If NULL has a separate business meaning, handle it explicitly.

Example:

```python
F.when(
    F.col("price").isNull(),
    "Unknown"
)
.when(
    F.col("price") < 10000,
    "Low"
)
.otherwise(
    "High"
)
```

This prevents a missing value from being accidentally classified
into a normal business category.

---

# 6. when() Without otherwise()

Example:

```python
F.when(
    F.col("status") == "F",
    "Fulfilled"
)
```

Result:

```text
status = F
→ Fulfilled

status != F
→ NULL
```

If no `otherwise()` is supplied, unmatched rows receive `NULL`.

---

# 7. Nested when()

A `when()` expression can contain another `when()`.

Example:

```python
F.when(
    F.col("status") == "O",

    F.when(
        F.col("total_price") > 100000,
        "High Priority Open"
    )
    .otherwise(
        "Standard Open"
    )
)
.otherwise(
    "Not Open"
)
```

Conceptually:

```text
Is order Open?
     ↓
    Yes
     ↓
Is value > 100000?
   ↓         ↓
  Yes        No
   ↓         ↓
High       Standard
Priority    Open

Otherwise
→ Not Open
```

Nested `when()` is useful for hierarchical decisions.

For simple categorisation, chained `when()` conditions are often easier
to read.

---

# 8. when() vs filter()

| Feature | `when()` | `filter()` |
|---|---|---|
| Purpose | Derive a value | Restrict rows |
| Creates/changes column value | Yes | No |
| Removes rows | No | Yes |
| Typical use | Flagging, categorising, mapping | Subsetting data |

### Easy way to remember

```text
when()
→ What VALUE should this row receive?

filter()
→ Should this ROW remain?
```

Using `when()` inside `select()` or `withColumn()` normally preserves
the existing rows.

---

# 9. SQL Equivalent

PySpark:

```python
F.when(
    condition,
    value
).otherwise(
    default_value
)
```

Conceptually corresponds to SQL:

```sql
CASE
    WHEN condition
    THEN value
    ELSE default_value
END
```

Multiple chained `when()` expressions correspond to multiple
SQL `WHEN` branches.

---

# 10. Typical Silver-Layer Uses

`when()` / `otherwise()` is useful for:

```text
Status mapping
Value segmentation
Business flags
Priority classification
NULL categorisation
Data-quality flags
Conditional standardisation
```

Examples:

```text
F → Fulfilled
O → Open
P → Pending
```

or:

```text
price < 10000
→ Low

price < 50000
→ Medium

otherwise
→ High
```

---

# 11. Silver-Layer Pattern

```text
Source columns
      ↓
Understand NULL behaviour
      ↓
Define business conditions
      ↓
Order conditions correctly
      ↓
when() / otherwise()
      ↓
Derive classification / flag
      ↓
Validate distribution
      ↓
Silver-ready attribute
```

---

# 12. Quick Decision Guide

```text
Need to map codes?
→ when()

Need to create a flag?
→ when()

Need range categorisation?
→ chained when()

Need hierarchical logic?
→ nested when()

Need default result?
→ otherwise()

Need NULL as separate category?
→ explicit isNull() branch

Need to remove rows?
→ filter(), not when()
```

---

# 13. Final S3.13 Understanding

```text
Condition
    ↓
TRUE?
 ┌───────┐
Yes      No / NULL
 ↓          ↓
Return    Next condition
value         ↓
          otherwise()
             ↓
         default value
```

Core principle:

```text
Top-to-bottom evaluation
        ↓
First TRUE condition wins
        ↓
Otherwise fallback
```

**S3.13 — Conditional Column Logic completed.**